In [2]:
import kagglehub
import os
import cv2
import numpy as np
import glob

print("1. Téléchargement du dataset Kaggle...")
path = kagglehub.dataset_download("emiliovenegas1/mammography-dataset-from-inbreast-mias-and-ddsm")

IMG_SIZE = 128
images = []
labels = []

image_files = glob.glob(os.path.join(path, '**', '*.*'), recursive=True)
image_files = [f for f in image_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"2. {len(image_files)} images trouvées. Chargement en mémoire (cela va prendre quelques minutes)...")

for img_path in image_files:
    label = os.path.basename(os.path.dirname(img_path))
    # Lecture en couleur (IMREAD_COLOR) car VGG16 s'attend à 3 canaux RGB
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)

    if img is not None:
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img)
        labels.append(label)

X = np.array(images)
y = np.array(labels)

print(f"✅ Chargement terminé ! Matrice des images : {X.shape}")


1. Téléchargement du dataset Kaggle...
Using Colab cache for faster access to the 'mammography-dataset-from-inbreast-mias-and-ddsm' dataset.
2. 26602 images trouvées. Chargement en mémoire (cela va prendre quelques minutes)...
✅ Chargement terminé ! Matrice des images : (26602, 128, 128, 3)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Encodage des labels...")
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Classes détectées : {le.classes_}")

# Division avec 'stratify' pour garder des proportions équilibrées
print("Séparation des données...")
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f"✅ Données d'entraînement : {len(X_train)} images")
print(f"✅ Données de test : {len(X_test)} images")


Encodage des labels...
Classes détectées : ['benign' 'malignant' 'normal']
Séparation des données...
✅ Données d'entraînement : 21281 images
✅ Données de test : 5321 images


In [5]:
import joblib
import numpy as np
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import gc # Garbage collector pour vider la RAM

print("1. Initialisation du Réseau Profond VGG16 (Le Fameux CNN)...")
cnn_extractor = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# NOUVELLE FONCTION OPTIMISÉE POUR NE PAS FAIRE PLANTER LA RAM COLAB
def extract_features_in_batches(images_array, batch_size=500):
    features_list = []
    total_images = len(images_array)

    for i in range(0, total_images, batch_size):
        # On ne prend qu'un petit bloc de 500 images à la fois
        batch = images_array[i:i+batch_size].astype('float32')
        batch = preprocess_input(batch)

        # Le CNN passe sur ce petit bloc
        features = cnn_extractor.predict(batch, verbose=0)
        features_list.append(features.reshape(features.shape[0], -1))

        print(f"Progression : {min(i+batch_size, total_images)} / {total_images} images extraites...")

        # Nettoyage manuel de la mémoire
        del batch; del features
        gc.collect()

    return np.vstack(features_list)

print("2. VGG16 : Analyse visuelle du jeu d'entraînement...")
X_train_features = extract_features_in_batches(X_train)

print("3. VGG16 : Analyse visuelle du jeu de test...")
X_test_features = extract_features_in_batches(X_test)

print("4. Machine Learning : Entraînement du SVM sur les caractéristiques...")
svm_classifier = SVC(kernel='rbf', probability=True, random_state=42)
svm_classifier.fit(X_train_features, y_train)

print("5. Verdict et Évaluation Finale !")
predictions = svm_classifier.predict(X_test_features)
acc = accuracy_score(y_test, predictions)

print(f"\n======================================")
print(f"🔥 Précision de l'Architecture Hybride : {acc * 100:.2f}% 🔥")
print(f"======================================")
print("\nRapport détaillé :")
print(classification_report(y_test, predictions, target_names=le.classes_))


1. Initialisation du Réseau Profond VGG16 (Le Fameux CNN)...
2. VGG16 : Analyse visuelle du jeu d'entraînement...
Progression : 500 / 21281 images extraites...
Progression : 1000 / 21281 images extraites...
Progression : 1500 / 21281 images extraites...
Progression : 2000 / 21281 images extraites...
Progression : 2500 / 21281 images extraites...
Progression : 3000 / 21281 images extraites...
Progression : 3500 / 21281 images extraites...
Progression : 4000 / 21281 images extraites...
Progression : 4500 / 21281 images extraites...
Progression : 5000 / 21281 images extraites...
Progression : 5500 / 21281 images extraites...
Progression : 6000 / 21281 images extraites...
Progression : 6500 / 21281 images extraites...
Progression : 7000 / 21281 images extraites...
Progression : 7500 / 21281 images extraites...
Progression : 8000 / 21281 images extraites...
Progression : 8500 / 21281 images extraites...
Progression : 9000 / 21281 images extraites...
Progression : 9500 / 21281 images extrait

In [6]:
from google.colab import files
import joblib

# On sauvegarde notre SVM sur la machine Colab
print("1. Écriture du fichier hybrid_svm.pkl...")
joblib.dump(svm_classifier, 'hybrid_svm.pkl')

# On force le téléchargement vers votre Windows
print("2. Téléchargement forcé vers votre PC (Acceptez si votre Navigateur le demande) :")
files.download('hybrid_svm.pkl')


1. Écriture du fichier hybrid_svm.pkl...
2. Téléchargement forcé vers votre PC (Acceptez si votre Navigateur le demande) :


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>